# Copper surface diffraction and incidence-angle response

A flat, ideal **Cu(001)** surface supplies a square lattice with **one atom per primitive surface cell**. The default beam is **30 keV**, **3 nm ideal FWHM**, incident at **4°** to the surface; the **256 × 256 Medipix3** detector has **55 µm pixels** and starts at **120 mm** camera length. The mechanical column remains inclined at **2°**, with a fixed **40 mm objective-to-sample distance**. Deflectors rock the beam around a fixed surface hit.

The calculation combines three explicit approximations: projected neutral-Cu atomic scattering factors set relative order weights; an abrupt complex potential step sets a smooth angle-dependent coherent-reflection envelope; the existing Gaussian surface adapter maps and propagates each order. This is not a dynamical Cu reflectivity calculation or a prediction of total backscattered-electron yield.

The **Incidence** slider updates the beam, diffraction image, geometry and intensity budget. The microscope side view and detector image share one figure. The GUI contains only the **Incidence**, **Camera z** and **Height** sliders. Objective aberrations are configured in `P`. The saved plane-wave estimates are distinct from the actual coherent intensity collected by the finite chip. Coherent Gaussian basis amplitudes are never treated as independent electron counts.

Laboratory Z is horizontal, X is height, and Y is lateral. The surface lies at X=0, with vacuum below. Results save to `output/copper/`, leaving earlier examples intact.


In [7]:
from pathlib import Path
import json, time
from dataclasses import replace
from types import SimpleNamespace
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
from temgym_core.components import Lens, KrivanekLens, Deflector, Plane, Detector, BraggSample
from temgym_core.aberrations import KrivanekCoeffs
from temgym_core.scattering import PotentialStepResponse, copper_scattering_factor
from temgym_core.coordinate_transforms import PlaneFrame
from temgym_core.source import gaussian_waist, decompose_gaussian_angular
from temgym_core.gaussian import FreeSpacePropagator
from temgym_core.run import run_to_end
from temgym_core.evaluate import gaussian_image, evaluate_gaussians, gpu_kernel_available
from temgym_core.plotting import trace_gaussian_basis, plot_reflection_column, export_ray_figure

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/temgym_core").exists())
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 11, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 120})

# Angles are degrees, lengths are metres, and potentials are volts/eV.
# The surface remains flat and the mechanical column inclination stays at 2°.
OBJECTIVE_TO_SAMPLE = 40e-3
P = dict(voltage=30_000., probe_fwhm=3e-9, column_inclination_deg=2., incidence_deg=4., rocking_deg=0.,
         azimuth_deg=0., camera_length=.120, objective_z=.310,
         working_distance=OBJECTIVE_TO_SAMPLE,
         aberrations=True, cs_mm=1., coma_um=10., coma_deg=0., astig_nm=0., astig_deg=0.,
         lattice_constant=.3615e-9, pixel_pitch=55e-6, detector_shape=(256,256),
         min_exit_deg=.25, broadening=8., extent_sigma=6.,
         inner_potential_v=21.2, absorption_ev=0., material="Cu(001)")
OUT = ROOT / "examples/sem_reflection/output/copper"
OUT.mkdir(parents=True, exist_ok=True)
P["sample_z"] = P["objective_z"] + OBJECTIVE_TO_SAMPLE
RUN_MILLION = True      # requires a working native Pallas GPU renderer
FULL_N = 640           # refined automatically at angles with fewer active orders
PREVIEW_N = 64
fs = FreeSpacePropagator()
print("JAX:", jax.__version__, jax.devices())

JAX: 0.10.2 [CudaDevice(id=0)]


## Column, fixed pivot and reference detector

The mechanical column inclination stays at **2°**. The initial incidence is **4°**; the incidence slider supplies additional fixed-pivot steering. The specimen remains horizontal. C1 images the source at 160 mm, C2 images that crossover at 265 mm, and the objective at 310 mm images it at the sample at 350 mm. The objective-to-sample distance is fixed at **40 mm**.

An affine seed followed by Newton refinement sets the actual surface intersection and incidence. The source width gives a 3 nm probe with the ideal objective. The objective settings in `P` apply Krivanek aberrations through the Gaussian complex action; steering is solved again after each change. Finite polepiece apertures are omitted.


In [8]:
alpha = np.deg2rad(P["incidence_deg"])
column_angle = np.deg2rad(P["column_inclination_deg"])
relative_angle = alpha-column_angle
# The column supplies the nominal inclination. The specimen remains flat.
pivot = jnp.zeros(3)
incident_axis = jnp.array([np.sin(alpha), 0., np.cos(alpha)])
column_axis = jnp.array([np.sin(column_angle),0.,np.cos(column_angle)])
column_frame = PlaneFrame(origin_xyz=pivot-P["sample_z"]*column_axis,
    u_xyz=jnp.array([np.cos(column_angle),0.,-np.sin(column_angle)]),v_xyz=jnp.array([0.,1.,0.]))
np.testing.assert_allclose(column_frame.normal_xyz,column_axis,atol=1e-15)
surface = PlaneFrame(origin_xyz=pivot,
    u_xyz=jnp.array([0.,0.,1.]),v_xyz=jnp.array([0.,1.,0.]))
specular = incident_axis - 2*(incident_axis @ surface.normal_xyz)*surface.normal_xyz
out_frame = PlaneFrame(origin_xyz=pivot,
    u_xyz=jnp.array([np.cos(alpha),0.,np.sin(alpha)]),v_xyz=jnp.array([0.,1.,0.]))
screen_frame = replace(out_frame, origin_xyz=pivot + P["camera_length"]*specular)
assert np.allclose(out_frame.normal_xyz, specular)
assert np.allclose(surface.normal_xyz,[-1.,0.,0.])
np.testing.assert_allclose(column_frame.to_world(jnp.array([0.,0.,P["sample_z"]])),pivot,atol=1e-15)
for frame in (column_frame,surface,out_frame,screen_frame):
    np.testing.assert_allclose(frame.basis_xyz.T @ frame.basis_xyz,np.eye(3),atol=1e-14)

probe_w = P["probe_fwhm"] / np.sqrt(2*np.log(2))
# The nominal beam is on the column axis. If nominal incidence is edited
# independently, compensate its small projection onto the beam-normal plane.
source_w = 1.5 * probe_w / np.cos(relative_angle)
source = gaussian_waist(z=0., voltage=P["voltage"], rms_size=1.5*probe_w/2)
source = source.derive(Q_inv=source.Q_inv.at[0,0].multiply(np.cos(relative_angle)**2),
                       amplitude=source.amplitude*np.sqrt(np.cos(relative_angle)))

# Refocus while retaining the same source and total 2/3 magnification.
# C1 images the source at 160 mm. C2 now images that crossover at 265 mm:
# its magnification is -3/4; the objective magnification is -8/9.
intermediate_image_z = .265
condenser2_f = 1/(1/(.220-.160) + 1/(intermediate_image_z-.220))
objective_f = 1/(1/(P["objective_z"]-intermediate_image_z) + 1/OBJECTIVE_TO_SAMPLE)

def objective_coeffs():
    if not P["aberrations"]:
        return KrivanekCoeffs()
    return KrivanekCoeffs(C30=P["cs_mm"]*1e-3,
        C21=P["coma_um"]*1e-6, phi21=np.deg2rad(P["coma_deg"]),
        C12=P["astig_nm"]*1e-9, phi12=np.deg2rad(P["astig_deg"]))


def column(kicks, coeffs=None):
    return (Lens(z=.080, focal_length=.040), Lens(z=.220, focal_length=condenser2_f),
            Deflector(z=.265, def_x=kicks[0], def_y=kicks[1]),
            Deflector(z=.285, def_x=kicks[2], def_y=kicks[3]),
            (Lens(z=P["objective_z"], focal_length=objective_f) if coeffs is None else
             KrivanekLens(z=P["objective_z"], focal_length=objective_f, coeffs=coeffs)),
            Plane(z=P["sample_z"]))

@jax.jit
def reference_state(kicks):
    ray = run_to_end(source,column(kicks),propagator=fs)
    return jnp.array([ray.x,ray.y,ray.dx,ray.dy])
steering_matrix = jax.jacfwd(reference_state)(jnp.zeros(4))

@jax.jit
def hit_and_angles(kicks, coeffs=KrivanekCoeffs()):
    centre = run_to_end(source, column(kicks, coeffs), propagator=fs)
    direction = jnp.array([centre.dx, centre.dy, 1.])
    direction = column_frame.basis_xyz @ (direction/jnp.linalg.norm(direction))
    origin_world = column_frame.to_world(jnp.array([centre.x, centre.y, centre.z]))
    hit, _ = surface.intersect_line(origin_world, direction)
    uv = surface.to_local(hit)[:2]
    grazing = jnp.arcsin(-direction @ surface.normal_xyz)
    azimuth = jnp.arctan2(direction @ surface.v_xyz, direction @ surface.u_xyz)
    return jnp.concatenate((uv, jnp.array([grazing, azimuth])))

scales = jnp.array([1e-5, 1e-5, 1e-3, 1e-3])
@jax.jit
def residual(kicks, target, coeffs):
    return (hit_and_angles(kicks, coeffs) - target) / scales
jacobian = jax.jit(jax.jacfwd(residual, argnums=0))

def solve_pivot(grazing_deg, azimuth_deg=0., initial=None, coeffs=KrivanekCoeffs()):
    target = jnp.array([0., 0., np.deg2rad(grazing_deg), np.deg2rad(azimuth_deg)])
    if not 0. < grazing_deg < 10.:
        raise ValueError("This paraxial example expects a grazing angle between 0 and 10 degrees.")
    # Build the small-angle seed in column coordinates, not laboratory angles.
    # Newton refines the actual surface intersection and exact direction.
    beta, azimuth = np.deg2rad(grazing_deg), np.deg2rad(azimuth_deg)
    desired_world = (jnp.cos(beta)*(jnp.cos(azimuth)*surface.u_xyz+jnp.sin(azimuth)*surface.v_xyz)
                     -jnp.sin(beta)*surface.normal_xyz)
    desired_column = column_frame.basis_xyz.T @ desired_world
    approximate_target = jnp.array([0.,0.,desired_column[0],desired_column[1]])
    kicks = (jnp.linalg.solve(steering_matrix,approximate_target-reference_state(jnp.zeros(4)))
             if initial is None else jnp.asarray(initial))
    history = []
    for _ in range(12):
        r = residual(kicks, target, coeffs)
        error = float(jnp.linalg.norm(r)); history.append(error)
        if error < 1e-10:
            return kicks, history
        delta = jnp.linalg.solve(jacobian(kicks, target, coeffs), r)
        for damping in (1., .5, .25, .125, .0625):
            trial = kicks - damping*delta
            if float(jnp.linalg.norm(residual(trial, target, coeffs))) < error:
                kicks = trial
                break
        else:
            raise RuntimeError("Newton step failed; reduce the requested rocking angle.")
    raise RuntimeError("Pivot solve did not converge.")

kicks, history = solve_pivot(P["incidence_deg"] + P["rocking_deg"], P["azimuth_deg"])
incident_column = column(kicks)
np.testing.assert_allclose(P["sample_z"]-incident_column[-2].z,OBJECTIVE_TO_SAMPLE,rtol=0,atol=1e-15)
print(f"Fixed objective-to-sample distance: {OBJECTIVE_TO_SAMPLE*1e3:.1f} mm")
print(f"Refocused focal lengths: C2={condenser2_f*1e3:.4f} mm, objective={objective_f*1e3:.4f} mm")
probe = run_to_end(source, incident_column, propagator=fs)
lab_fwhm_x = np.sqrt(2/(float(probe.k)*float(probe.Q_inv[0,0].imag)))*np.sqrt(2*np.log(2))
actual_fwhm = lab_fwhm_x*np.cos(relative_angle)
np.testing.assert_allclose(actual_fwhm, P["probe_fwhm"], rtol=1e-8)
print(f"Probe FWHM: {actual_fwhm*1e9:.6f} nm; surface footprint: {actual_fwhm/np.sin(float(hit_and_angles(kicks)[2]))*1e9:.2f} nm")
print("D1x, D1y, D2x, D2y (mrad):", np.asarray(kicks)*1e3)
objective_beam = run_to_end(source,incident_column[:-1],propagator=fs)
objective_offset_m = float(objective_beam.x)
np.testing.assert_allclose(objective_beam.r_xy,
    -OBJECTIVE_TO_SAMPLE*probe.d_xy,rtol=1e-8,atol=1e-12)
objective_world = column_frame.to_world(jnp.array([0.,0.,P["objective_z"]]))
np.testing.assert_allclose(jnp.linalg.norm(pivot-objective_world),OBJECTIVE_TO_SAMPLE,rtol=0,atol=1e-15)
if abs(P["incidence_deg"]-P["column_inclination_deg"])+abs(P["rocking_deg"])+abs(P["azimuth_deg"]) < 1e-12:
    np.testing.assert_allclose(kicks,0.,atol=1e-12)
    np.testing.assert_allclose(objective_offset_m,0.,atol=1e-12)
print(f"Objective centroid offset: {objective_offset_m*1e3:.4f} mm; zero-aberration reference")
print("Newton residual history:",history)


rocking = []
for offset in np.linspace(-.2, .2, 9):
    setting, convergence = solve_pivot(P["incidence_deg"]+offset, .05)
    measured = np.asarray(hit_and_angles(setting))
    rocking.append([offset, np.linalg.norm(measured[:2]), *np.asarray(setting)])
rocking = np.asarray(rocking)
assert rocking[:,1].max() < 1e-12
print(f"Rocking verification: maximum surface pivot error {rocking[:,1].max()*1e12:.3g} pm")

# Decompose through the IDEAL column: nonlinear optics must never be inverted
# using the linear ABCD decomposition. Propagate those same source packets
# forward through the aberrated objective, retaining their action and centres.
def incident_packets(n):
    bundle = decompose_gaussian_angular(source, num_across=n, through=column(kicks),
        broadening=P["broadening"], extent_sigma=P["extent_sigma"])
    return bundle, run_to_end(bundle, incident_column, propagator=fs)


Fixed objective-to-sample distance: 40.0 mm
Refocused focal lengths: C2=25.7143 mm, objective=21.1765 mm
Probe FWHM: 3.000000 nm; surface footprint: 43.01 nm
D1x, D1y, D2x, D2y (mrad): [-3.10406840e+01  0.00000000e+00  1.21155757e-14  0.00000000e+00]
Objective centroid offset: -1.3968 mm; zero-aberration reference
Newton residual history: [0.021246895285126355, 1.575788111107812e-08, 1.3877787807814457e-14]
Rocking verification: maximum surface pivot error 1.36e-08 pm


The detector orientation is fixed normal to the initial specular direction. Changing incidence automatically translates its centre onto the new reflected chief ray. Use **Camera z** and **Height** to position the detector independently. This separates the angular response of the surface from detector acceptance.


## Cu lattice, complex reflection action and intensity budget

The fcc lattice constant is $a=0.3615$ nm. The primitive Cu(001) surface lattice along the two $\langle110\rangle$ directions has period $a_s=a/\sqrt{2}=0.2556$ nm and one Cu atom per cell. The adapter retains the zeroth-Laue-zone row $(h,m)=(0,-6),\ldots,(0,6)$. [Cu(001) structure and clean 1×1 RHEED](https://www.ritsumei.ac.jp/acd/re/src/memoirs/no15/no15_08.pdf).

Projected top-layer amplitudes are proportional to the neutral-Cu **Lobato–Van Dyck electron scattering factor** at the tangential reciprocal frequency $|m|/a_s$. Parameters and frequency convention follow [abTEM's implementation](https://github.com/abTEM/abTEM/blob/main/abtem/parametrizations/data/lobato.json). A fixed reference row $m=-256,\ldots,256$ sets a unit-flux partition. This normalization is phenomenological: it does not derive absolute Cu reflectivity. The retained subset is never renormalized after evanescence, grazing cutoff or detector clipping. Normal momentum dependence, deeper layers and other Laue zones are omitted.

For the angular envelope, an attractive abrupt potential step has electron potential energy $-e(V_0+iW)$. With $E^*=E(1+E/(2mc^2))$ and $U=V_0+iW$, define

$$p_n=\sqrt{E^*}\sin\alpha,\qquad p'_n=\sqrt{p_n^2+U(1+E/mc^2)+U^2/(2mc^2)},$$
$$r(\alpha)=\frac{p_n-p'_n}{p_n+p'_n},\qquad R_{\rm step}=|r|^2.$$

The decaying complex-root branch is used. $V_0=21.2$ V is a [published Cu mean inner potential](https://cpl.iphy.ac.cn/article/id/33824). $W=0$ eV by default; a nonzero absorptive optical potential is editable but not fitted. The abrupt surface profile is itself an approximation, especially near grazing incidence. This envelope is not a sum of all backscattered electrons or a dynamical Bragg rocking curve.

`BraggSample.coefficients_at` supplies $c_m(\alpha)=r(\alpha)w_m$ for each incident angular packet. Its equivalent complex action is

$$\Delta S_m=-\frac{i}{k}\log c_m=\frac{\arg c_m}{k}-\frac{i}{k}\log|c_m|,$$

so $\exp(ik\Delta S_m)=c_m$. **An intensity fraction $R$ therefore gives a field multiplier $\sqrt R$, not $R$.** This action depends on incident angle. It is sampled across the angular Gaussian basis, not differentiated as a spatial absorption map; the response is approximated as constant across each narrow packet.

The surface adapter conserves tangential momentum and selects the outward elastic branch. Gaussian envelopes use the surface-to-screen Jacobian and paraxial free-space propagation. Exit angles below 0.25° are excluded. The saved metadata reports (1) smooth-step $R$, (2) the retained propagating-order fraction for a plane wave, and (3) detector-integrated coherent intensity divided by incident flux. The latter uses the actual finite probe and chip; it is not a sum of packet intensities.


In [9]:
# Cu(001): one atom per primitive square surface cell, along <110>.
surface_period = P["lattice_constant"] / np.sqrt(2)
lateral_period = surface_period
orders = np.column_stack((np.zeros(13,dtype=int),np.arange(-6,7)))
# Projected top-layer factors: evaluate at tangential reciprocal frequencies.
# Normalize over a fixed, much larger reference row, NEVER after angular or
# detector clipping. This is an explicit phenomenological flux partition.
reference_m = np.arange(-256,257)
reference_f = np.asarray(copper_scattering_factor(abs(reference_m)/surface_period*1e-10))
order_norm = np.linalg.norm(reference_f)
coefficients = np.asarray(copper_scattering_factor(abs(orders[:,1])/surface_period*1e-10),dtype=complex)/order_norm
response = PotentialStepResponse(inner_potential_v=P["inner_potential_v"],absorption_ev=P["absorption_ev"])
bragg = BraggSample(q_xy=jnp.asarray(2*np.pi*orders/surface_period),
                    coefficients=jnp.asarray(coefficients),angular_response=response)
assert np.sum(abs(coefficients)**2) <= 1.+1e-12

def reflect_packets(incident, orders_hm=orders, coeff=coefficients, *, apply_response=True):
    """Notebook-local kinematic surface adapter; returns active packets + audit.

    Intersect each packet with the surface, add its lattice phase, and
    transport it to the common outgoing plane. The local paraxial envelope
    projection retains displaced centres produced by objective aberrations.
    """
    incident = incident.to_vector()
    count,norder = incident.x.size,len(orders_hm)
    hm = jnp.asarray(orders_hm)
    vi = jnp.stack((incident.dx,incident.dy,jnp.ones_like(incident.x)),axis=-1)
    vi /= jnp.linalg.norm(vi,axis=-1,keepdims=True)
    vi = vi @ column_frame.basis_xyz.T
    incoming_normal = vi @ surface.normal_xyz
    tu = (vi @ surface.u_xyz)[:,None] + 2*jnp.pi*hm[None,:,0]/(surface_period*incident.k[:,None])
    tv = (vi @ surface.v_xyz)[:,None] + 2*jnp.pi*hm[None,:,1]/(lateral_period*incident.k[:,None])
    radicand = 1-tu**2-tv**2
    normal = jnp.sqrt(jnp.maximum(radicand,1e-20))
    forward = (tu*jnp.dot(surface.u_xyz,out_frame.normal_xyz)
               +tv*jnp.dot(surface.v_xyz,out_frame.normal_xyz)
               +normal*jnp.dot(surface.normal_xyz,out_frame.normal_xyz))
    valid = ((radicand>np.sin(np.deg2rad(P["min_exit_deg"]))**2)
             &(incoming_normal[:,None]<0)&(forward>0))
    host_valid = np.asarray(valid)
    parents,order_indices = np.nonzero(host_valid)
    # Prune BEFORE constructing Gaussian matrices: shallow incidence can have
    # a million specular packets but many millions of evanescent candidates.
    beam = jax.tree.map(lambda x:x[parents],incident)
    directions_in = vi[parents]
    tu,tv,normal,forward = (x[parents,order_indices] for x in (tu,tv,normal,forward))
    vo = tu[:,None]*surface.u_xyz+tv[:,None]*surface.v_xyz+normal[:,None]*surface.normal_xyz
    if apply_response:
        grazing = jnp.arcsin(jnp.clip(-incoming_normal,-1.,1.))
        selected = replace(bragg,coefficients=jnp.asarray(coeff))
        c = selected.coefficients_at(grazing,incident.voltage)[parents,order_indices]
    else:
        c = jnp.asarray(coeff)[order_indices]
    # First reach the individual surface hit in the incoming coordinates.
    # Work relative to the pivot to avoid subtracting the 350 mm column
    # origin when retaining picometre-scale optical path differences.
    offsets = jnp.stack((beam.x,beam.y,beam.z-P["sample_z"]),axis=-1)
    origins = pivot + offsets @ column_frame.basis_xyz.T
    hits,_ = surface.intersect_line(origins,directions_in)
    dz_in = (hits-origins) @ column_frame.normal_xyz
    beam = fs(beam,dz_in)
    local_out = out_frame.to_local(hits)
    surface_uv = surface.to_local(hits)[:,:2]
    grating_action = jnp.sum(2*jnp.pi*hm[order_indices]*surface_uv/
                            jnp.array([surface_period,lateral_period]),axis=-1)/beam.k
    identity = jnp.eye(3)
    pi = identity-directions_in[:,:,None]*surface.normal_xyz[None,None,:]/incoming_normal[parents,None,None]
    po = identity-vo[:,:,None]*out_frame.normal_xyz[None,None,:]/forward[:,None,None]
    transverse_out = out_frame.basis_xyz[:,:2]
    A = jnp.einsum('ij,bjk,bkl,lm->bim',transverse_out.T,po,pi,column_frame.basis_xyz[:,:2])
    inv_A = jnp.linalg.inv(A)
    Q = jnp.swapaxes(inv_A,-1,-2) @ beam.Q_inv @ inv_A
    jac = jnp.abs(jnp.linalg.det(A))
    slopes = (vo @ transverse_out)/forward[:,None]
    reflected = beam.derive(x=local_out[:,0],y=local_out[:,1],z=local_out[:,2],
        dx=slopes[:,0],dy=slopes[:,1],Q_inv=Q,
        amplitude=beam.amplitude*c/jnp.sqrt(jac),
        pathlength=beam.pathlength+dz_in-local_out[:,2]+grating_action)
    # Restore the relative axial carrier between the two posed frames above;
    # fs itself omits that carrier. All packets then share outgoing z=0.
    reflected = fs(reflected,-reflected.z)
    elastic_error = float(jnp.max(jnp.abs(jnp.sum(vo**2,axis=1)-1))) if len(parents) else 0.
    return reflected,dict(incident=int(count),candidates=int(count*norder),active=int(len(parents)),
        elastic_error=elastic_error,
        per_order={str(tuple(map(int,h))):int(host_valid[:,i].sum()) for i,h in enumerate(orders_hm)})


def make_reflected(n):
    _, arriving = incident_packets(n)
    reflected, audit = reflect_packets(arriving)
    if not audit["active"]:
        raise ValueError("No reflected packets satisfy the angular acceptance.")
    detector_beams = run_to_end(reflected, (Plane(z=P["camera_length"]),), propagator=fs)
    return detector_beams, audit


def make_full_reflected():
    # Keep the million-packet promise even when only specular is propagating.
    _, small_audit = make_reflected(PREVIEW_N)
    orders_per_parent = small_audit["active"]/PREVIEW_N**2
    n = max(FULL_N,int(np.ceil(np.sqrt(1_050_000/orders_per_parent))))
    result, audit = make_reflected(n)
    while audit["active"] < 1_000_000:
        n = int(np.ceil(n*1.1))
        result,audit = make_reflected(n)
    audit["num_across"] = n
    return result,audit

# The single-chip detector lives in the outgoing frame, not laboratory Z.
detector = Detector(z=P["camera_length"], shape=P["detector_shape"],
                    pixel_size=(P["pixel_pitch"],)*2)
GPU = gpu_kernel_available()
method = "spatial" if GPU else "scan"
print("Renderer:", method)

Renderer: spatial


## Verify the basis and preview the pattern

Angular quadrature reconstructs the source field after native propagation through the complete column. Refining the basis holds individual packet widths fixed. The accelerated detector evaluator is also compared with an independent scan at selected detector points.

A specular-only render is compared with an analytic reflected Gaussian. This separately checks the local paraxial scattering approximation. Nominal incidence is supplied mechanically, so the chief ray is on-axis within the column; any extra electronic tilt is measured relative to that axis.


In [10]:
x = np.linspace(-2, 2, 61)*P["probe_fwhm"]
probe_grid = SimpleNamespace(shape=x.shape, coords=jnp.asarray(np.column_stack((x,np.zeros_like(x)))))
reference_field = np.asarray(evaluate_gaussians(probe, probe_grid, method="scan"))
test_bundle = decompose_gaussian_angular(source, num_across=64, through=incident_column,
    broadening=P["broadening"], extent_sigma=P["extent_sigma"])
test_probe = run_to_end(test_bundle, incident_column, propagator=fs)
basis_field = np.asarray(evaluate_gaussians(test_probe, probe_grid, method="scan"))
basis_error = np.linalg.norm(basis_field-reference_field)/np.linalg.norm(reference_field)
assert basis_error < 1e-5, basis_error
preview_beams, preview_audit = make_reflected(PREVIEW_N)
preview = np.asarray(gaussian_image(preview_beams, detector, samples_per_pixel=4,
                                   method=method, tile_beams=32))
assert preview_audit["elastic_error"] < 1e-12
# Independent backend comparison at a sparse, real detector grid.
check_detector = replace(detector, shape=(12,12), pixel_size=(.001,.001))
a = np.asarray(evaluate_gaussians(preview_beams, check_detector, method="scan"))
b = np.asarray(evaluate_gaussians(preview_beams, check_detector, method=method))
renderer_error = np.linalg.norm(a-b)/np.linalg.norm(a)
assert renderer_error < 1e-5, renderer_error
print(f"Complex probe reconstruction error: {basis_error:.3g}")
print(f"Spatial / scan field difference: {renderer_error:.3g}")
print(preview_audit)

# A unit-reflecting specular-only surface tests geometry independently of the response.
# It must recover the analytic reflected round Gaussian.
mirror, _ = reflect_packets(test_probe, np.array([[0,0]]), np.array([1.+0j]),apply_response=False)
mirror_screen = run_to_end(mirror,(Plane(z=P["camera_length"]),),propagator=fs)
mirror_image = np.asarray(gaussian_image(mirror_screen,detector,samples_per_pixel=4,method=method))
# This independent round-beam reference applies to nominal, unrocked incidence.
if abs(P["rocking_deg"])+abs(P["azimuth_deg"]) < 1e-12:
    # Reflect the tilted central beam and express its envelope normal to
    # nominal reflection. The projection depends on tilt relative to the column axis.
    inverse_projection = jnp.diag(jnp.array([-1/np.cos(relative_angle),1.]))
    reference_mirror = probe.derive(z=0.,dx=0.,dy=0.,
        Q_inv=inverse_projection @ probe.Q_inv @ inverse_projection,
        amplitude=probe.amplitude/np.sqrt(np.cos(relative_angle)))
    reference_mirror = run_to_end(reference_mirror,(Plane(z=P["camera_length"]),),propagator=fs)
    reference_image = np.asarray(gaussian_image(reference_mirror,detector,samples_per_pixel=4,method=method))
    specular_error = float(np.linalg.norm(mirror_image-reference_image)/np.linalg.norm(reference_image))
    # This checks the local paraxial model separately from numerical quadrature.
    # Nominal inclination is mechanical; only relative electronic tilt enters.
    paraxial_specular_tolerance = .001 + 4*relative_angle**2
    assert specular_error < paraxial_specular_tolerance, specular_error
    print(f"Specular-only image / analytic Gaussian difference: {specular_error:.3g}")
else:
    specular_error = None


# Subsequent renders use the selected objective. These coefficients are
# explicit JAX arguments in the steering solve, never mutable JIT closures.
kicks,_ = solve_pivot(P["incidence_deg"]+P["rocking_deg"], P["azimuth_deg"], coeffs=objective_coeffs())
incident_column = column(kicks,objective_coeffs())
probe = run_to_end(source,incident_column,propagator=fs)
preview_beams,preview_audit = make_reflected(PREVIEW_N)
preview = np.asarray(gaussian_image(preview_beams,detector,samples_per_pixel=4,method=method,tile_beams=32))


Complex probe reconstruction error: 4.84e-09
Spatial / scan field difference: 7.4e-09
{'incident': 4096, 'candidates': 53248, 'active': 20480, 'elastic_error': 0.0, 'per_order': {'(0, -6)': 0, '(0, -5)': 0, '(0, -4)': 0, '(0, -3)': 0, '(0, -2)': 4096, '(0, -1)': 4096, '(0, 0)': 4096, '(0, 1)': 4096, '(0, 2)': 4096, '(0, 3)': 0, '(0, 4)': 0, '(0, 5)': 0, '(0, 6)': 0}}
Specular-only image / analytic Gaussian difference: 0.00479


## Full coherent render: over one million outgoing Gaussians

This cell really propagates the angular decomposition through the SEM, splits it into reflected orders and sums the outgoing complex Gaussian fields. No ray histogram or independent-spot intensity approximation is substituted. The large run fails explicitly without a working GPU kernel; set `RUN_MILLION=False` for a portable preview.

The convergence comparison changes angular quadrature density while holding the physical source, packet width and angular extent fixed. A second comparison refines pixel integration. These checks assess numerical sampling, not the adequacy of the kinematic physics.

In [11]:
if RUN_MILLION and not GPU:
    raise RuntimeError("The million-packet run needs the GPU renderer. Set RUN_MILLION=False for a preview.")
t0 = time.perf_counter()
if RUN_MILLION:
    beams, audit = make_full_reflected()
    assert audit["active"] >= 1_000_000, "Increase FULL_N for these physical settings."
    image = np.asarray(gaussian_image(beams, detector, samples_per_pixel=4,
                                     method="spatial", tile_beams=32))
else:
    beams, audit, image = preview_beams, preview_audit, preview
render_seconds = time.perf_counter()-t0

# A denser preview should approach the full calculation.
medium_beams, medium_audit = make_reflected(128)
medium_image = np.asarray(gaussian_image(medium_beams, detector, samples_per_pixel=4,
                                         method=method, tile_beams=32))
relative = lambda a,b: float(np.linalg.norm(a-b)/np.linalg.norm(b))
convergence = dict(preview_to_final=relative(preview,image), medium_to_final=relative(medium_image,image))
refined_pixels = np.asarray(gaussian_image(medium_beams, detector, samples_per_pixel=5,
                                          method=method, tile_beams=32))
convergence["pixel_4_to_5"] = relative(medium_image,refined_pixels)
assert convergence["medium_to_final"] < .01, convergence
assert convergence["pixel_4_to_5"] < .005, convergence
assert np.isfinite(image).all() and image.max() > 0
incident_flux = float(np.pi*abs(source.amplitude)**2/(source.k*np.sqrt(np.linalg.det(np.asarray(source.Q_inv.imag)))))
np.testing.assert_allclose(incident_flux,1.,rtol=1e-12)
report = dict(parameters=P, incident_flux=incident_flux,
    step_reflectivity=float(response.reflectivity(np.deg2rad(P["incidence_deg"]+P["rocking_deg"]),P["voltage"])),
    detector_fraction=float(image.sum()/incident_flux), sample_pose="flat", column_pose="inclined",
    objective_to_sample_m=OBJECTIVE_TO_SAMPLE, condenser2_focal_length_m=condenser2_f,
    objective_focal_length_m=objective_f, deflector_kicks_rad=np.asarray(kicks).tolist(),objective_offset_m=objective_offset_m, full_n=audit.get("num_across",PREVIEW_N),
    audit=audit, render_seconds=render_seconds, basis_error=float(basis_error),
    renderer_error=float(renderer_error), specular_error=specular_error, convergence=convergence,
    model_detector_flux=float(image.sum()), max_pivot_error_m=float(rocking[:,1].max()),
    jax_version=jax.__version__, device=str(jax.devices()[0]))
(OUT/"run.json").write_text(json.dumps(report,indent=2)+"\n")
np.savez_compressed(OUT/"detector.npz", intensity=image, preview=preview, medium=medium_image,
                    parameters=json.dumps(P), rocking=rocking)
print(json.dumps(report, indent=2))

{
  "parameters": {
    "voltage": 30000.0,
    "probe_fwhm": 3e-09,
    "column_inclination_deg": 2.0,
    "incidence_deg": 4.0,
    "rocking_deg": 0.0,
    "azimuth_deg": 0.0,
    "camera_length": 0.12,
    "objective_z": 0.31,
    "working_distance": 0.04,
    "aberrations": true,
    "cs_mm": 1.0,
    "coma_um": 10.0,
    "coma_deg": 0.0,
    "astig_nm": 0.0,
    "astig_deg": 0.0,
    "lattice_constant": 3.615e-10,
    "pixel_pitch": 5.5e-05,
    "detector_shape": [
      256,
      256
    ],
    "min_exit_deg": 0.25,
    "broadening": 8.0,
    "extent_sigma": 6.0,
    "inner_potential_v": 21.2,
    "absorption_ev": 0.0,
    "material": "Cu(001)",
    "sample_z": 0.35
  },
  "incident_flux": 1.0000000000000002,
  "step_reflectivity": 0.0012103172157034144,
  "detector_fraction": 0.0011060406731865384,
  "sample_pose": "flat",
  "column_pose": "inclined",
  "objective_to_sample_m": 0.04,
  "condenser2_focal_length_m": 0.025714285714285717,
  "objective_focal_length_m": 0.0211764705

## Incidence and detector

Use **Incidence** (0.5–6°), **Camera z**, and **Height** below. Sliders render on release. The incidence control retraces the column and rebuilds the reflected basis; any cached full render is invalidated. The mechanical column stays at 2° and the fixed surface pivot is preserved.

Changing incidence automatically centres the detector on the specular ray. Camera z and height can then be adjusted independently. Detector orientation remains fixed. The figure shows the microscope side view and detector image.

Objective aberrations remain configured in `P`: defaults are 1 mm spherical, 10 µm coma and zero astigmatism. The colour scale is fixed to the initial image peak, so angular signal changes remain visible. Intensity budgets are retained in `camera_state` and saved metadata. To produce and save a full render from code, call `request_camera_update(full=True, save=True)` with the GPU renderer available.


In [12]:
import ipywidgets as widgets
from IPython.display import display, clear_output

reference_peak = float(image.max())  # held fixed during angle/camera changes
full_camera_cache = {"beam":beams if RUN_MILLION else None,"audit":audit}


def current_angle():
    return P["incidence_deg"]+P["rocking_deg"]


def refresh_incident(angle_deg):
    """Physically rock the incident beam at a fixed surface pivot."""
    global kicks,incident_column,probe,test_bundle,test_probe,preview_at_sample,camera_preview_audit
    global representative_directions,path_weights,trace,incident_envelope
    P["rocking_deg"] = float(angle_deg)-P["incidence_deg"]
    kicks,_ = solve_pivot(float(angle_deg),P["azimuth_deg"],coeffs=objective_coeffs())
    incident_column = column(kicks,objective_coeffs())
    probe = run_to_end(source,incident_column,propagator=fs)
    test_bundle,test_probe = incident_packets(PREVIEW_N)
    preview_at_sample,camera_preview_audit = reflect_packets(test_probe)
    directions,weights = [],[]
    for hm,coefficient in zip(orders,coefficients):
        group,group_audit = reflect_packets(test_probe,np.array([hm]),np.array([coefficient]))
        if group_audit["active"]:
            index = int(jnp.argmax(abs(group.amplitude)))
            directions.append(np.asarray(out_frame.basis_xyz @ jnp.array([group.dx[index],group.dy[index],1.])))
            weights.append(float(abs(group.amplitude[index])**2))
    representative_directions = jnp.asarray(directions)
    path_weights = np.asarray(weights)
    trace = trace_gaussian_basis(column(kicks),source_w,voltage=P["voltage"])
    # The side-view envelope follows the local Gaussian action.
    rows=[]
    beam=source
    for component in incident_column:
        distance=float(component.z-beam.z)
        zs=jnp.linspace(0.,distance,100)
        segment=fs(beam,zs)
        covariance=jnp.linalg.inv(segment.Q_inv.imag)/(2*segment.k[...,None,None])
        rows.extend(np.column_stack((segment.z,segment.x,2*jnp.sqrt(covariance[:,0,0]))))
        beam=component(fs(beam,distance))
    incident_envelope=np.asarray(rows)


refresh_incident(current_angle())


def camera_geometry(z_mm,height_mm):
    """Independent laboratory Z/X translations; sensor orientation is fixed."""
    centre_world = pivot+jnp.array([height_mm*1e-3,0.,z_mm*1e-3])
    centre_local = out_frame.to_local(centre_world)
    distance = float(centre_local[2])
    if not np.isfinite(np.asarray(centre_local)).all() or distance <= 0:
        raise ValueError("The detector must remain in front of the reflected reference plane.")
    # Grid applies its centre in (row, column) order BEFORE the default
    # row flip, despite its XY type annotation. Convert physical (u, v)
    # explicitly: rendered x = centre[1], rendered y = -centre[0].
    assert detector.rotation == 0. and not detector.flip_y
    camera = replace(detector,z=distance,
                     centre=(-float(centre_local[1]),float(centre_local[0])))
    np.testing.assert_allclose(np.asarray(camera.coords).mean(axis=0),
                               np.asarray(centre_local[:2]),atol=1e-14)
    frame = replace(out_frame,origin_xyz=centre_world)
    hits,_ = frame.intersect_line(pivot,representative_directions)
    starts = jnp.broadcast_to(pivot,hits.shape)
    paths = np.asarray(jnp.stack((starts,hits),axis=1))[:,:,[0,2]]
    half_height = camera.pixel_size[1]*camera.shape[1]/2
    edge = frame.to_world(jnp.array([[-half_height,0.,0.],[half_height,0.,0.]]))
    return dict(detector=camera,frame=frame,paths=paths,segment=np.asarray(edge)[:,[0,2]],
                centre_uv_m=np.asarray(centre_local[:2]),
                z_mm=float(z_mm),height_mm=float(height_mm),distance_m=distance,
                centre_world_xyz_m=np.asarray(centre_world).tolist())


def camera_figures(geometry,intensity):
    camera = geometry["detector"]
    fig = plt.figure(figsize=(13,5.2),layout="constrained",facecolor="white")
    grid = fig.add_gridspec(1,2,width_ratios=(1.65,1),wspace=.12)
    ax = fig.add_subplot(grid[0,0])
    axd = fig.add_subplot(grid[0,1])
    plot_reflection_column(incident_column,trace,sample_z=P["sample_z"],
        outgoing_paths_xz=geometry["paths"],detector_segment_xz=geometry["segment"],
        surface_angle_rad=0.,column_frame=column_frame,
        component_labels=("C1","C2","D1","D2","Objective","Reference plane"),
        order_weights=path_weights,
        title="Microscope side view",specimen_label="Cu(001)",
        show_section_labels=False,incident_envelope=incident_envelope,ax=ax)
    ax.set_title("Microscope side view",loc="left",fontsize=14,pad=12,color="#21374C")
    ax.set_xlabel("Horizontal position (mm)",fontsize=10)
    ax.set_ylabel("Height (mm)",fontsize=10)
    xc,yc = map(np.asarray,camera.coords_1d)
    # Plot coordinates relative to the sensor centre; its world position is
    # controlled by the sliders and shown in the side view.
    screen_image = intensity[np.argsort(yc),:][:,np.argsort(xc)].T
    xc,yc = xc-geometry["centre_uv_m"][0],yc-geometry["centre_uv_m"][1]
    extent = [yc.min()*1e3-camera.pixel_size[0]*500,yc.max()*1e3+camera.pixel_size[0]*500,
              xc.min()*1e3-camera.pixel_size[1]*500,xc.max()*1e3+camera.pixel_size[1]*500]
    axd.imshow(np.maximum(screen_image/reference_peak,1e-5),origin="lower",extent=extent,
                   aspect="equal",cmap="magma",norm=LogNorm(1e-5,1),interpolation="nearest")
    axd.set(xlabel="Lateral position (mm)",ylabel="Height (mm)",facecolor="#000004")
    axd.set_title("Detector image",loc="left",fontsize=14,pad=12,color="#21374C")
    axd.xaxis.label.set_size(10); axd.yaxis.label.set_size(10)
    axd.tick_params(colors="#566777",labelsize=9,length=3)
    for spine in axd.spines.values():
        spine.set_visible(False)
    # Label exact chief-ray rod intersections; the image remains the coherent
    # sum of the full angular basis. Hide labels outside the physical sensor.
    vi = column_frame.basis_xyz @ jnp.array([probe.dx,probe.dy,1.])
    vi = vi/jnp.linalg.norm(vi)
    for hm in orders:
        tu = float(vi @ surface.u_xyz) + 2*np.pi*hm[0]/(surface_period*float(probe.k))
        tv = float(vi @ surface.v_xyz) + 2*np.pi*hm[1]/(lateral_period*float(probe.k))
        rad = 1-tu**2-tv**2
        if rad <= np.sin(np.deg2rad(P["min_exit_deg"]))**2:
            continue
        vo = tu*surface.u_xyz + tv*surface.v_xyz + np.sqrt(rad)*surface.normal_xyz
        hit, distance = geometry["frame"].intersect_line(pivot,vo)
        uv = np.asarray(geometry["frame"].to_local(hit))*1e3
        if float(distance)>0 and extent[0]<uv[1]<extent[1] and extent[2]<uv[0]<extent[3]:
            label = f"{hm[1]:g}"
            axd.annotate(label,(uv[1],uv[0]),xytext=(0,22),textcoords="offset points",
                         ha="center",fontsize=8,color="#B8CCD9",clip_on=True)
    return fig


# Plane-wave angular budget: percentages of incident flux, not packet counts.
def angle_budget(angles_deg):
    angles = np.deg2rad(np.asarray(angles_deg))
    step = np.asarray(response.reflectivity(angles,P["voltage"]))
    tangential = orders[:,1]*float(probe.wavelength)/lateral_period
    azimuth = np.deg2rad(P["azimuth_deg"])
    normal_squared = 1-(np.cos(angles[...,None])*np.cos(azimuth))**2-(np.cos(angles[...,None])*np.sin(azimuth)+tangential)**2
    valid = normal_squared > np.sin(np.deg2rad(P["min_exit_deg"]))**2
    retained = step*np.sum(valid*abs(coefficients)**2,axis=-1)
    return step,retained


def specular_height(z_mm,angle_deg):
    return -np.tan(np.deg2rad(angle_deg))*z_mm


default_z_mm = float(screen_frame.origin_xyz[2]-pivot[2])*1e3
default_height_mm = float(screen_frame.origin_xyz[0]-pivot[0])*1e3
slider_style={"description_width":"125px"}
angle_slider=widgets.FloatSlider(value=current_angle(),min=.5,max=6.,step=.05,description="Incidence (°)",
    readout_format=".2f",continuous_update=False,style=slider_style,layout=widgets.Layout(width="100%"))
z_slider=widgets.FloatSlider(value=default_z_mm,min=60.,max=800.,step=.5,description="Camera z (mm)",
    readout_format=".1f",continuous_update=False,style=slider_style,layout=widgets.Layout(width="100%"))
height_slider=widgets.FloatSlider(value=default_height_mm,min=-100.,max=30.,step=.1,description="Height (mm)",
    readout_format=".2f",continuous_update=False,style=slider_style,layout=widgets.Layout(width="100%"))
figure_output=widgets.Output(layout=widgets.Layout(width="100%"))
camera_state={"suspend":False,"busy":False,"error":None}


def show_camera(geometry,intensity,quality,packet_audit,elapsed=None,save=False):
    captured = float(intensity.sum()/incident_flux)
    step,retained = map(float,angle_budget(current_angle()))
    fig=camera_figures(geometry,intensity)
    try:
        pose={key:geometry[key] for key in ("z_mm","height_mm","distance_m","centre_world_xyz_m")}
        budget=dict(incidence_deg=current_angle(),step_fraction=step,retained_plane_wave_fraction=retained,
                    detector_coherent_fraction=captured,incident_flux=incident_flux)
        if save:
            metadata=dict(parameters=P,camera_pose=pose,quality=quality,audit=packet_audit,flux_budget=budget)
            export_ray_figure(fig,OUT/"microscope_detector",metadata={"title":"Cu(001) microscope and detector",**metadata})
            np.savez_compressed(OUT/"camera_view.npz",intensity=intensity,camera_pose=json.dumps(pose),
                                parameters=json.dumps(P),quality=quality,flux_budget=json.dumps(budget))
            (OUT/"camera_view.json").write_text(json.dumps(metadata,indent=2)+"\n")
            degrees=np.linspace(.5,6.,600); envelope,orders_fraction=angle_budget(degrees)
            np.savetxt(OUT/"reflectivity.csv",np.column_stack((degrees,envelope,orders_fraction)),delimiter=",",
                       header="incidence_deg,step_fraction,retained_plane_wave_fraction",comments="")
        with figure_output:
            clear_output(wait=True); display(fig)
        camera_state.update(geometry=geometry,intensity=intensity,quality=quality,audit=packet_audit,
                            flux_budget=budget,error=None)
    finally:
        plt.close(fig)


def update_camera(*,full=False,save=False):
    geometry=camera_geometry(z_slider.value,height_slider.value)
    t0=time.perf_counter()
    if full:
        if not GPU:raise RuntimeError("The million-Gaussian render requires the GPU kernel.")
        if full_camera_cache["beam"] is None:
            full_camera_cache["beam"],full_camera_cache["audit"]=make_full_reflected()
        base,packet_audit=full_camera_cache["beam"],full_camera_cache["audit"]
        quality="Full render"
    else:
        base,packet_audit=preview_at_sample,camera_preview_audit
        quality="Preview"
    propagated=run_to_end(base,(Plane(z=geometry["distance_m"]),),propagator=fs)
    result=np.asarray(gaussian_image(propagated,geometry["detector"],samples_per_pixel=4,method=method,tile_beams=32))
    show_camera(geometry,result,quality,packet_audit,time.perf_counter()-t0,save=save)


def request_camera_update(*,full=False,save=False,angle_changed=False):
    if camera_state["suspend"] or camera_state["busy"]:return
    camera_state["busy"]=True
    controls=(angle_slider,z_slider,height_slider)
    for control in controls:control.disabled=True
    try:
        if angle_changed:
            # Invalidate before rebuilding so a failed refresh cannot reuse stale full beams.
            full_camera_cache.update(beam=None,audit=None)
            refresh_incident(angle_slider.value)
            height_slider.value=specular_height(z_slider.value,angle_slider.value)
        update_camera(full=full,save=save)
    except Exception as error:
        camera_state["error"]=str(error)
        with figure_output:
            clear_output(wait=True)
            print(f"Render failed: {error}")
    finally:
        camera_state["busy"]=False
        for control in controls:control.disabled=False


angle_slider.observe(lambda change:request_camera_update(angle_changed=True),names="value")
z_slider.observe(lambda change:request_camera_update(),names="value")
height_slider.observe(lambda change:request_camera_update(),names="value")
camera_controls=widgets.VBox([angle_slider,z_slider,height_slider,figure_output],
    layout=widgets.Layout(width="100%",max_width="1300px"))
display(camera_controls)
show_camera(camera_geometry(default_z_mm,default_height_mm),image,
            "Full render" if RUN_MILLION else "Preview",audit,save=True)


## Scope and next physical refinements

The Cu lattice and atomic factors are physically specified. The allocation of the smooth-step reflection envelope across projected top-layer orders remains a phenomenological approximation. It omits normal momentum dependence of atomic factors, bulk layer interference, dynamical multiple scattering, surface relaxation, thermal diffuse scattering and inelastic backscatter. In particular, the percentage plotted here must not be interpreted as total SEM backscatter yield.

The complex-action interface can accept a calibrated complex angular response later without changing Gaussian propagation. Quantitative Cu reflection would require a surface potential and dynamical scattering calculation (or measured complex order coefficients). The objective uses the existing `KrivanekLens` complex-action path. The source basis is constructed through the ideal column, then propagated through the aberrated lens; the surface adapter retains displaced packet centres and their lattice/carrier phases. The side-view envelope is the local Gaussian approximation, while the detector image uses a coherent packet sum. Finite apertures, partial coherence and calibrated aberration coefficients are still needed for hardware predictions.

The combined view is exported as `microscope_detector.png/.svg/.pdf`, alongside the `reflectivity.csv` angle table and numerical metadata.
